# ORBIT-SAGE: evaluate a saved model and run the empty-scene test

In [ ]:
!pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

## Saved model

In [ ]:
!ls -lh /content/drive/MyDrive/orbit_runs/ship_final/weights/

## Dataset

In [ ]:
import os
from getpass import getpass
if not os.path.isdir('/content/HRSC2016_extracted'):
    os.environ['KAGGLE_API_TOKEN'] = getpass('Kaggle token: ')
    !pip install -q kaggle
    !kaggle datasets download -d guofeng/hrsc2016 -p /content/
    !unzip -q /content/hrsc2016.zip -d /content/HRSC2016
    !apt-get install -y unrar -qq
    !unrar x -idq "/content/HRSC2016/hrsc2016_dataset/HRSC2016.part01.rar" /content/HRSC2016_extracted/

## Scripts

In [ ]:
%%writefile train_full.py
# ORBIT-SAGE: convert HRSC2016 to YOLO, train, tune threshold, export ONNX
import argparse, json, math, random, shutil, sys
import xml.etree.ElementTree as ET
from pathlib import Path
IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp')

def obb_to_hbb(cx, cy, w, h, ang):
    c, s = (math.cos(ang), math.sin(ang))
    xs, ys = ([], [])
    for dx, dy in ((-w / 2, -h / 2), (w / 2, -h / 2), (w / 2, h / 2), (-w / 2, h / 2)):
        xs.append(cx + dx * c - dy * s)
        ys.append(cy + dx * s + dy * c)
    return (min(xs), min(ys), max(xs), max(ys))

def parse_annotation(xml_path):
    root = ET.parse(xml_path).getroot()
    W = int(root.find('Img_SizeWidth').text)
    H = int(root.find('Img_SizeHeight').text)
    boxes = []
    for o in root.findall('HRSC_Objects/HRSC_Object'):
        g = lambda k: float(o.find(k).text)
        x0, y0, x1, y1 = obb_to_hbb(g('mbox_cx'), g('mbox_cy'), g('mbox_w'), g('mbox_h'), g('mbox_ang'))
        x0, y0, x1, y1 = (max(x0, 0), max(y0, 0), min(x1, W), min(y1, H))
        if x1 - x0 > 2 and y1 - y0 > 2:
            boxes.append(((x0 + x1) / 2 / W, (y0 + y1) / 2 / H, (x1 - x0) / W, (y1 - y0) / H))
    return boxes

def convert(root, out, split_mode='random', seed=42, neg_ratio=0.5):
    from PIL import Image
    root, out = (Path(root), Path(out))
    BAD = ('segment', 'mask', 'visual', 'vis', 'label')

    def ok(p):
        return not any((b in '/'.join(p.parts[len(root.parts):]).lower() for b in BAD))
    xmls = [p for p in root.rglob('*.xml') if 'Annotations' in p.parts and ok(p)]
    full = [p for p in xmls if 'FullDataSet' in p.parts]
    xmls = sorted(full or xmls)
    if not xmls:
        sys.exit(f'No Annotations/*.xml under {root}')
    cands = {}
    for p in root.rglob('*'):
        if p.suffix.lower() in IMG_EXTS and out not in p.parents and ok(p):
            cands.setdefault(p.stem, []).append(p)

    def pick(stem):
        c = cands.get(stem, [])
        pref = [p for p in c if any((k in p.parts for k in ('AllImages', 'JPEGImages')))]
        return ((pref or c or [None])[0], bool(pref))
    samples, imgs, nonpref = ({}, {}, 0)
    for x in xmls:
        ip, pref = pick(x.stem)
        if ip is None:
            continue
        try:
            b = parse_annotation(x)
        except Exception as e:
            print('skip', x.name, e)
            continue
        if b:
            samples[x.stem] = b
            imgs[x.stem] = ip
            nonpref += not pref
    print(f'{len(samples)} usable images (of {len(xmls)} annotations); {nonpref} not from AllImages/JPEGImages')
    if nonpref:
        print('WARNING: some images come from other folders - check them:', [str(imgs[k]) for k in list(samples) if not any((q in imgs[k].parts for q in ('AllImages', 'JPEGImages')))][:5])
    stems = sorted(samples)
    official = {}
    for name in ('train', 'val', 'test'):
        f = next(iter(root.rglob(f'ImageSets/{name}.txt')), None)
        if f:
            official[name] = [l.strip() for l in f.read_text().split() if l.strip() in samples]
    if split_mode == 'official' and len(official) == 3:
        splits = official
    else:
        if split_mode == 'official':
            print('official ImageSets not found -> random split')
        random.Random(seed).shuffle(stems)
        n = len(stems)
        a, b = (int(n * 0.8), int(n * 0.9))
        splits = {'train': stems[:a], 'val': stems[a:b], 'test': stems[b:]}
    if out.exists():
        shutil.rmtree(out)
    for sp, items in splits.items():
        (out / 'images' / sp).mkdir(parents=True)
        (out / 'labels' / sp).mkdir(parents=True)
        for st in items:
            Image.open(imgs[st]).convert('RGB').save(out / 'images' / sp / f'{st}.jpg', quality=95)
            (out / 'labels' / sp / f'{st}.txt').write_text(''.join((f'0 {a:.6f} {b:.6f} {c:.6f} {d:.6f}\n' for a, b, c, d in samples[st])))
        nneg = 0
        rng = random.Random(seed + len(sp))
        for st in items:
            if rng.random() >= neg_ratio:
                continue
            im = Image.open(out / 'images' / sp / f'{st}.jpg')
            Wd, Hd = im.size
            boxes = [((cx - w / 2) * Wd, (cy - h / 2) * Hd, (cx + w / 2) * Wd, (cy + h / 2) * Hd) for cx, cy, w, h in samples[st]]
            for _ in range(40):
                ts = int(rng.uniform(0.3, 0.5) * min(Wd, Hd))
                x0 = rng.randint(0, Wd - ts)
                y0 = rng.randint(0, Hd - ts)
                if all((x0 + ts < b[0] - 10 or x0 > b[2] + 10 or y0 + ts < b[1] - 10 or (y0 > b[3] + 10) for b in boxes)):
                    im.crop((x0, y0, x0 + ts, y0 + ts)).save(out / 'images' / sp / f'{st}_neg.jpg', quality=95)
                    (out / 'labels' / sp / f'{st}_neg.txt').write_text('')
                    nneg += 1
                    break
        print(f'{sp}: {len(items)} images + {nneg} empty-scene tiles')
    yaml = out / 'dataset.yaml'
    yaml.write_text(f'path: {out.resolve()}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: ship\n')
    return yaml

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    i = ix * iy
    u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - i
    return i / u if u > 0 else 0

def prf_at(per_image, thr):
    tp = fp = fn = 0
    for preds, gts in per_image:
        used = set()
        for p in sorted((p for p in preds if p[4] >= thr), key=lambda p: -p[4]):
            best, bj = (0.5, None)
            for j, g in enumerate(gts):
                if j not in used and iou(p[:4], g) >= best:
                    best, bj = (iou(p[:4], g), j)
            if bj is None:
                fp += 1
            else:
                tp += 1
                used.add(bj)
        fn += len(gts) - len(used)
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    return (tp, fp, fn, P, R, 2 * P * R / (P + R) if P + R else 0.0)

def collect(model, data_root, split, imgsz):
    from PIL import Image
    out = []
    for ip in sorted((Path(data_root) / 'images' / split).glob('*.jpg')):
        W, H = Image.open(ip).size
        gts = []
        for l in (Path(data_root) / 'labels' / split / f'{ip.stem}.txt').read_text().splitlines():
            _, cx, cy, w, h = map(float, l.split())
            gts.append(((cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H))
        r = model.predict(str(ip), conf=0.001, imgsz=imgsz, verbose=False)[0]
        preds = [(*b.xyxy[0].tolist(), float(b.conf[0])) for b in r.boxes]
        out.append((preds, gts))
    return out

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--root', default='/content/HRSC2016_extracted')
    ap.add_argument('--data', default='/content/hrsc_yolo')
    ap.add_argument('--project', default='/content/drive/MyDrive/orbit_runs')
    ap.add_argument('--name', default='ship_final')
    ap.add_argument('--split', default='random', choices=['random', 'official'])
    ap.add_argument('--model', default='yolov8s.pt')
    ap.add_argument('--epochs', type=int, default=100)
    ap.add_argument('--imgsz', type=int, default=800)
    ap.add_argument('--batch', type=int, default=8)
    ap.add_argument('--patience', type=int, default=25)
    ap.add_argument('--negatives', type=float, default=0.5, help='share of images that get a ship-free tile (0 = off)')
    ap.add_argument('--convert-only', action='store_true')
    ap.add_argument('--allow-cpu', action='store_true')
    a, _ = ap.parse_known_args()
    yaml = convert(a.root, a.data, a.split, neg_ratio=a.negatives)
    if a.convert_only:
        return
    import torch
    if not torch.cuda.is_available() and (not a.allow_cpu):
        sys.exit('NO GPU. Runtime > Change runtime type > T4 GPU, restart, rerun.')
    from ultralytics import YOLO
    model = YOLO(a.model)
    model.train(data=str(yaml), epochs=a.epochs, imgsz=a.imgsz, batch=a.batch, patience=a.patience, project=a.project, name=a.name, exist_ok=True, cos_lr=True, seed=0, save_period=5, workers=2, plots=True, device=0 if torch.cuda.is_available() else 'cpu')
    save_dir = Path(model.trainer.save_dir)
    best = save_dir / 'weights' / 'best.pt'
    print('BEST WEIGHTS:', best)
    m = YOLO(str(best))
    val_curve = collect(m, a.data, 'val', a.imgsz)
    grid = [round(t, 2) for t in [0.05 * i for i in range(1, 19)]]
    f1s = {t: prf_at(val_curve, t)[5] for t in grid}
    conf = max(f1s, key=f1s.get)
    tp, fp, fn, P, R, F1 = prf_at(collect(m, a.data, 'test', a.imgsz), conf)
    r = m.val(data=str(yaml), split='test', imgsz=a.imgsz, conf=0.001, plots=True)
    res = {'model': a.model, 'imgsz': a.imgsz, 'split': a.split, 'epochs_requested': a.epochs, 'test_mAP50': float(r.box.map50), 'test_mAP50_95': float(r.box.map), 'recommended_conf_from_val': conf, 'test_at_that_conf': {'TP': tp, 'FP': fp, 'FN': fn, 'precision': P, 'recall': R, 'f1': F1}}
    (save_dir / 'detector_metrics.json').write_text(json.dumps(res, indent=2))
    print(json.dumps(res, indent=2))
    m.export(format='onnx', imgsz=a.imgsz)
    print('Saved in', save_dir)
if __name__ == '__main__':
    main()


In [ ]:
%%writefile evaluate_pipeline.py
# ORBIT-SAGE: packet builder and Problem 7 evaluation on the test set
import argparse, io, json, time
from pathlib import Path
from PIL import Image
LINK_BPS = 50000
HIGH, MID, FLOOR = (0.9, 0.6, 0.25)

def jpeg_bytes(img, q):
    b = io.BytesIO()
    img.convert('RGB').save(b, 'JPEG', quality=q)
    return len(b.getvalue())

def build_packet_bytes(img, dets):
    top = max((d[4] for d in dets), default=0.0)
    meta = len(json.dumps({'ships': [[round(v, 1) for v in d] for d in dets]}).encode()) + 120
    if not dets:
        return ('M0', meta)
    crops = sum((jpeg_bytes(img.crop(tuple((int(v) for v in d[:4]))), 85) for d in dets))
    if top >= HIGH:
        return ('M1', meta + crops)
    thumb = img.copy()
    thumb.thumbnail((256, 256))
    if top >= MID:
        return ('M2', meta + crops + jpeg_bytes(thumb, 60))
    return ('M3', meta + jpeg_bytes(img, 50))

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    i = ix * iy
    u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - i
    return i / u if u > 0 else 0

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--data', default='/content/hrsc_yolo')
    ap.add_argument('--weights', default='/content/drive/MyDrive/orbit_runs/ship_final/weights/best.pt')
    ap.add_argument('--imgsz', type=int, default=800)
    ap.add_argument('--out', default='/content/drive/MyDrive/orbit_runs/ship_final/pipeline_results.json')
    a, _ = ap.parse_known_args()
    from ultralytics import YOLO
    model = YOLO(a.weights)
    rows = []
    tot = dict(raw=0, jpeg50=0, packet=0, gt=0, found=0, inf_ms=0.0)
    for ip in sorted((Path(a.data) / 'images' / 'test').glob('*.jpg')):
        img = Image.open(ip)
        W, H = img.size
        gts = []
        for l in (Path(a.data) / 'labels' / 'test' / f'{ip.stem}.txt').read_text().splitlines():
            _, cx, cy, w, h = map(float, l.split())
            gts.append(((cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H))
        t = time.perf_counter()
        r = model.predict(str(ip), conf=FLOOR, imgsz=a.imgsz, verbose=False)[0]
        inf = (time.perf_counter() - t) * 1000
        dets = [(*b.xyxy[0].tolist(), float(b.conf[0])) for b in r.boxes]
        mode, pbytes = build_packet_bytes(img, dets)
        raw = ip.stat().st_size
        j50 = jpeg_bytes(img, 50)
        found = len(gts) if mode == 'M3' else sum((any((iou(d[:4], g) >= 0.5 for d in dets)) for g in gts))
        rows.append(dict(image=ip.name, mode=mode, raw_bytes=raw, packet_bytes=pbytes, reduction=1 - pbytes / raw, ships=len(gts), ships_delivered=found, inference_ms=inf))
        tot['raw'] += raw
        tot['jpeg50'] += j50
        tot['packet'] += pbytes
        tot['gt'] += len(gts)
        tot['found'] += found
        tot['inf_ms'] += inf
    n = len(rows)
    neg = [r for r in rows if r['ships'] == 0]
    modes = {m: sum((r['mode'] == m for r in rows)) for m in ('M0', 'M1', 'M2', 'M3')}
    res = dict(images=n, mode_counts=modes, data_reduction_vs_raw=1 - tot['packet'] / tot['raw'], data_reduction_vs_jpeg50=1 - tot['packet'] / tot['jpeg50'], mean_packet_kb=tot['packet'] / n / 1024, mean_raw_kb=tot['raw'] / n / 1024, information_preserved=tot['found'] / max(tot['gt'], 1), empty_scene_images=len(neg), empty_scene_false_alarm_rate=sum((r['mode'] != 'M0' for r in neg)) / len(neg) if neg else None, empty_scene_data_reduction=1 - sum((r['packet_bytes'] for r in neg)) / sum((r['raw_bytes'] for r in neg)) if neg else None, mean_inference_ms=tot['inf_ms'] / n, mean_link_seconds_raw=tot['raw'] / n / LINK_BPS, mean_link_seconds_packet=tot['packet'] / n / LINK_BPS, note='raw = original JPEG size; link at 50 KB/s; inference measured on this machine')
    Path(a.out).write_text(json.dumps(dict(summary=res, per_image=rows), indent=2))
    print(json.dumps(res, indent=2))
if __name__ == '__main__':
    main()


In [ ]:
%%writefile make_empty_tiles.py
# ORBIT-SAGE: build ship-free test crops
import argparse, random, shutil
from pathlib import Path
from PIL import Image

def free_windows(W, H, boxes, size, tries, rng, margin=24):
    out = []
    for _ in range(tries):
        x0 = rng.randint(0, W - size)
        y0 = rng.randint(0, H - size)
        x1, y1 = (x0 + size, y0 + size)
        if all((bx1 + margin < x0 or bx0 - margin > x1 or by1 + margin < y0 or (by0 - margin > y1) for bx0, by0, bx1, by1 in boxes)):
            out.append((x0, y0, x1, y1))
    return out

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--data', default='/content/hrsc_yolo')
    ap.add_argument('--out', default='/content/hrsc_empty')
    ap.add_argument('--size', type=int, default=416)
    ap.add_argument('--per_image', type=int, default=2)
    ap.add_argument('--seed', type=int, default=42)
    a, _ = ap.parse_known_args()
    rng = random.Random(a.seed)
    out = Path(a.out)
    shutil.rmtree(out, ignore_errors=True)
    (out / 'images' / 'test').mkdir(parents=True)
    (out / 'labels' / 'test').mkdir(parents=True)
    n = 0
    for ip in sorted((Path(a.data) / 'images' / 'test').glob('*.jpg')):
        img = Image.open(ip).convert('RGB')
        W, H = img.size
        if min(W, H) < a.size:
            continue
        boxes = []
        for l in (Path(a.data) / 'labels' / 'test' / f'{ip.stem}.txt').read_text().splitlines():
            _, cx, cy, w, h = map(float, l.split())
            boxes.append(((cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H))
        for k, win in enumerate(free_windows(W, H, boxes, a.size, 60, rng)[:a.per_image]):
            name = f'{ip.stem}_empty{k}'
            img.crop(win).save(out / 'images' / 'test' / f'{name}.jpg', quality=95)
            (out / 'labels' / 'test' / f'{name}.txt').write_text('')
            n += 1
    print('ship-free tiles:', n, '->', out)
if __name__ == '__main__':
    main()


In [ ]:
%%writefile make_example.py
# ORBIT-SAGE: export one real packet example per mode
import argparse, io, json, shutil, statistics
from pathlib import Path
from PIL import Image, ImageDraw
FLOOR = 0.25

def jpg(img, q):
    b = io.BytesIO()
    img.convert('RGB').save(b, 'JPEG', quality=q)
    return b.getvalue()

def build(img, dets, mode, outdir, tag):
    sizes = {}
    meta = json.dumps({'ships': [[round(v, 1) for v in d] for d in dets]}).encode()
    (outdir / f'{tag}_meta.json').write_bytes(meta)
    sizes['metadata (JSON)'] = len(meta) + 120
    if mode in ('M1', 'M2'):
        for k, d in enumerate(dets):
            c = jpg(img.crop(tuple((int(v) for v in d[:4]))), 85)
            (outdir / f'{tag}_crop{k + 1}.jpg').write_bytes(c)
            sizes[f'crop {k + 1}'] = len(c)
    if mode == 'M2':
        t = img.copy()
        t.thumbnail((256, 256))
        c = jpg(t, 60)
        (outdir / f'{tag}_thumb.jpg').write_bytes(c)
        sizes['thumbnail 256px'] = len(c)
    if mode == 'M3':
        c = jpg(img, 50)
        (outdir / f'{tag}_full_q50.jpg').write_bytes(c)
        sizes['full image JPEG-50'] = len(c)
    return sizes

def overlay(img, dets):
    o = img.convert('RGB').copy()
    d = ImageDraw.Draw(o)
    for x0, y0, x1, y1, c in dets:
        d.rectangle([x0, y0, x1, y1], outline=(255, 40, 40), width=max(3, img.width // 300))
        d.text((x0 + 4, max(0, y0 - 14)), f'ship {c:.2f}', fill=(255, 255, 0))
    return o

def pick(rows, mode):
    cand = [r for r in rows if r['mode'] == mode and r['ships'] in (1, 2, 3)]
    if not cand:
        cand = [r for r in rows if r['mode'] == mode]
    if not cand:
        return None
    med = statistics.median((r['packet_bytes'] for r in cand))
    return min(cand, key=lambda r: abs(r['packet_bytes'] - med))

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--data', default='/content/hrsc_yolo')
    ap.add_argument('--weights', default='/content/drive/MyDrive/orbit_runs/ship_final/weights/best.pt')
    ap.add_argument('--results', default='/content/drive/MyDrive/orbit_runs/ship_final/pipeline_results.json')
    ap.add_argument('--out', default='/content/orbit_examples')
    a, _ = ap.parse_known_args()
    from ultralytics import YOLO
    model = YOLO(a.weights)
    rows = json.loads(Path(a.results).read_text())['per_image']
    out = Path(a.out)
    shutil.rmtree(out, ignore_errors=True)
    out.mkdir(parents=True)
    report = {}
    for mode in ('M1', 'M2', 'M3'):
        r = pick(rows, mode)
        if r is None:
            continue
        ip = Path(a.data) / 'images' / 'test' / r['image']
        img = Image.open(ip)
        res = model.predict(str(ip), conf=FLOOR, imgsz=800, verbose=False)[0]
        dets = [(*b.xyxy[0].tolist(), float(b.conf[0])) for b in res.boxes]
        tag = f'{mode}_{ip.stem}'
        shutil.copy(ip, out / f'{tag}_original.jpg')
        overlay(img, dets).save(out / f'{tag}_detections.jpg', quality=92)
        sizes = build(img, dets, mode, out, tag)
        raw = ip.stat().st_size
        total = sum(sizes.values())
        report[mode] = dict(image=ip.name, raw_bytes=raw, packet_bytes=total, pieces=sizes, reduction=1 - total / raw, top_conf=max((d[4] for d in dets), default=None), ships_detected=len(dets), ships_in_ground_truth=r['ships'])
        print(mode, ip.name, 'raw', raw, 'packet', total, f'-{100 * (1 - total / raw):.0f}%', sizes)
    (out / 'examples.json').write_text(json.dumps(report, indent=2))
    shutil.make_archive('/content/orbit_examples', 'zip', out)
    print('zip: /content/orbit_examples.zip')
if __name__ == '__main__':
    main()


## Rebuild the split

In [ ]:
!python train_full.py --root /content/HRSC2016_extracted --data /content/hrsc_yolo --negatives 0 --convert-only
!ls /content/hrsc_yolo/images/test | wc -l

## Evaluate the test set

In [ ]:
!python evaluate_pipeline.py --out /content/pipeline_results.json

## Empty-scene test

In [ ]:
!python make_empty_tiles.py
!ls /content/hrsc_empty/images/test | wc -l

In [ ]:
!python evaluate_pipeline.py --data /content/hrsc_empty --out /content/pipeline_empty.json

## Examples

In [ ]:
!python make_example.py

## Download

In [ ]:
from google.colab import files
files.download('/content/pipeline_results.json')
files.download('/content/pipeline_empty.json')